In [ ]:
from argparse import Namespace

In [ ]:
import pandas as pd

In [ ]:
#from eap.graph import Graph
from eap import graph

In [ ]:
from eap.utils import tokenize_plus

In [ ]:
from transformer_lens import HookedTransformer

In [ ]:
from ablation_utils.utils import make_hooks

In [ ]:
from simple_eap import mic_score

In [ ]:
import importlib

In [ ]:
importlib.reload(graph)

<module 'eap.graph' from '/mounts/work/sgerstner/RW_functionalities/EAP-IG/src/eap/graph.py'>

In [ ]:
DEVICE='cuda:1'

In [ ]:
my_graph = graph.Graph.from_pt('../RW_functionalities_results/full_graph_with_subnodes.pt')

In [ ]:
#subnodes_df = my_graph.subnodes_to_pandas('../RW_functionalities_results/subnodes.csv')
subnodes_df = pd.from_csv('../RW_functionalities_results/subnodes.csv')

In [ ]:
subnodes_df.head(40)

,name,layer,node_name,subnode_index,score
0,input,0,input,NaN,0.000000e+00
1,a0.h0,0,a0.h0,NaN,0.000000e+00
2,a0.h1,0,a0.h1,NaN,0.000000e+00
3,a0.h2,0,a0.h2,NaN,0.000000e+00
4,a0.h3,0,a0.h3,NaN,0.000000e+00
5,a0.h4,0,a0.h4,NaN,0.000000e+00
6,a0.h5,0,a0.h5,NaN,0.000000e+00
7,a0.h6,0,a0.h6,NaN,0.000000e+00
8,a0.h7,0,a0.h7,NaN,0.000000e+00
9,a0.h8,0,a0.h8,NaN,0.000000e+00


In [ ]:
sorted_df = subnodes_df.sort_values("score", ascending=False).head(49)
print(sorted_df)

          name  layer node_name  subnode_index     score
1549  m31.8581     31       m31         8581.0  1.510321
1405  m30.9996     30       m30         9996.0  1.103545
527        m12     12       m12            NaN  0.726136
1424   a31.h12     31   a31.h12            NaN  0.663149
1339   a30.h10     30   a30.h10            NaN  0.620403
1320  m29.7261     29       m29         7261.0  0.571579
685        m16     16       m16            NaN  0.542899
1444       m31     31       m31            NaN  0.520416
981        m23     23       m23            NaN  0.467276
1363   m30.732     30       m30          732.0  0.456894
1345   a30.h16     30   a30.h16            NaN  0.377761
1043  m24.7935     24       m24         7935.0  0.374789
1306  m29.4131     29       m29         4131.0  0.373894
566        m13     13       m13            NaN  0.350341
1428   a31.h16     31   a31.h16            NaN  0.278034
810        m19     19       m19            NaN  0.271658
941        m22     22       m22

In [ ]:
sorted_filtered_df = subnodes_df[subnodes_df["subnode_index"].notna()].sort_values("score", ascending=False)
print(sorted_filtered_df.head(40))

           name  layer node_name  subnode_index     score
1549   m31.8581     31       m31         8581.0  1.510321
1405   m30.9996     30       m30         9996.0  1.103545
1320   m29.7261     29       m29         7261.0  0.571579
1363    m30.732     30       m30          732.0  0.456894
1043   m24.7935     24       m24         7935.0  0.374789
1306   m29.4131     29       m29         4131.0  0.373894
1489   m31.3498     31       m31         3498.0  0.193198
1377   m30.3705     30       m30         3705.0  0.114631
1506   m31.4568     31       m31         4568.0  0.114614
1148   m26.5558     26       m26         5558.0  0.085032
612    m14.4516     14       m14         4516.0  0.066492
1253   m28.6703     28       m28         6703.0  0.039890
1524   m31.5973     31       m31         5973.0  0.037797
1551   m31.8666     31       m31         8666.0  0.037575
1090   m25.6139     25       m25         6139.0  0.027810
1541   m31.7081     31       m31         7081.0  0.024053
1155   m26.684

## Evaluating via ablations

In [ ]:
model = HookedTransformer.from_pretrained_no_processing(#with predefined neurons, our ablation code is implementation-invariant
        'allenai/OLMo-7B-0424-hf',
        device=DEVICE,
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Loaded pretrained model allenai/OLMo-7B-0424-hf into HookedTransformer


In [ ]:
sequence = "Yesterday (21 December) the Government announced a package of support for hospitality and leisure businesses that are losing trade because of the O"

In [ ]:
tokens, attention_mask, input_lengths, n_pos = tokenize_plus(model, [sequence])

In [ ]:
args = Namespace(
    gate='-', post='+',
    activation_location='mlp.hook_post',
    intervention_type='zero_ablation',
)

In [ ]:
logits = model(tokens, attention_mask=attention_mask)
score = mic_score(logits, clean_logits=None, input_lengths=input_lengths)
score_list = [score]

In [ ]:
hook_list = []
for row in sorted_filtered_df.iterrows():
    hook_list.extend(
        make_hooks(
            args,
            layer=row.layer, neuron=int(row.subnode_index),
            mean_value=0.0,
        )
    )
    with model.hooks(hook_list):
        logits = model(tokens, attention_mask=attention_mask)
        score = mic_score(logits, clean_logits=None, input_lengths=input_lengths)
        score_list.append(score)

In [ ]:
print(score)